# Análisis de préstamos de libros en bibliotecas de Madrid con Spark

¿Cuáles son los libros más prestados en las bibliotecas públicas de Madrid? En este notebook responderemos a esa pregunta procesando un dataset real de millones de registros de préstamos bibliotecarios — esta vez usando **Apache Spark** (DataFrames y Spark SQL) en lugar de MapReduce.

**Dataset:** Préstamos de las bibliotecas públicas de Madrid (2024) en formato CSV: [https://datos.madrid.es/dataset/212700-0-bibliotecas-prestamos-historico/information](https://datos.madrid.es/dataset/212700-0-bibliotecas-prestamos-historico/information)

## Contenido
1. Configurar la sesión Spark con el clúster
2. Descargar el CSV y subirlo a HDFS
3. Explorar el esquema y los datos
4. Contar préstamos por título con la API DataFrame
5. Contar préstamos por título con Spark SQL
6. Filtrar por tipo de soporte (solo libros) con DataFrame API
7. Filtrar por tipo de soporte con Spark SQL
8. Ejercicio 1: Tipos de soporte
9. Ejercicio 2: Préstamos de otros tipos de soportes
10. Ejercicio 3: Analizar préstamos por sucursal (códigos y nombres reales)
11. Ejercicio 4: Analizar préstamos a lo largo del tiempo y visualizar con matplotlib
12. Cerrar la sesión de Spark
---

## 1. Configurar la sesión Spark con el clúster

Creamos una `SparkSession` conectada al clúster Spark desplegado con Docker Compose. El master escucha en `spark://spark-master:7077` y cada worker tiene 1 core y 1 GB de memoria.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Libros_Spark") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"App ID: {spark.sparkContext.applicationId}")

---

## 2. Descargar el CSV y subirlo a HDFS

Descargamos el fichero CSV en local, lo subimos a HDFS (para que todos los workers de Spark puedan acceder a él) y lo cargamos como un DataFrame de Spark indicando el delimitador `;` y que la primera fila contiene la cabecera.

In [ ]:
# Si ya realizásteis este paso en el ejercicio de MapReduce, no es necesario repetirlo

import os

os.makedirs("data", exist_ok=True)
!wget -qO data/prestamos_bibliotecas_madrid_2024.csv "http://192.168.70.194/prestamos_bibliotecas_madrid_2024.csv"
!wc -l data/prestamos_bibliotecas_madrid_2024.csv

# Subir el CSV a HDFS para que los workers de Spark puedan acceder al fichero
!hadoop fs -mkdir -p libros/input
!hadoop fs -put -f data/prestamos_bibliotecas_madrid_2024.csv libros/input
!hadoop fs -ls -h libros/input

In [ ]:
# Leer el CSV desde HDFS como DataFrame de Spark

df = spark.read.csv(
    "hdfs://master:8020/user/jovyan/libros/input/prestamos_bibliotecas_madrid_2024.csv",
    sep=";",
    header=True,
    inferSchema=True,
    multiLine=True
)

print(f"Filas cargadas: {df.count():,}")

---

## 3. Explorar el esquema y los datos

Spark infiere automáticamente el tipo de cada columna. Veamos el esquema resultante y una muestra de los datos.

In [ ]:
# Esquema inferido por Spark
df.printSchema()

In [ ]:
# Primeras 10 filas
df.show(10, truncate=False)

In [ ]:
# Valores distintos del tipo de soporte (prcocs)
df.select("prcocs").distinct().show(50, truncate=False)

---

## 4. Contar préstamos por título con la API DataFrame

Equivalente al MapReduce: agrupamos por título del libro (`tititu`) y contamos las ocurrencias. Con Spark esto se hace en una sola línea encadenando transformaciones.

In [ ]:
from pyspark.sql.functions import col, count, desc

# Agrupar por título, contar préstamos y ordenar de mayor a menor
prestamos_por_titulo = (
    df.groupBy("tititu")
      .agg(count("*").alias("prestamos"))
      .orderBy(desc("prestamos"))
)

prestamos_por_titulo.show(20, truncate=False)

---

## 5. Contar préstamos por título con Spark SQL

Registramos el DataFrame como una vista temporal y lanzamos la misma consulta en SQL puro. Spark optimiza ambos enfoques de la misma manera internamente (Catalyst optimizer).

In [ ]:
# Registrar el DataFrame como vista temporal SQL
df.createOrReplaceTempView("prestamos")

# Consulta SQL equivalente
spark.sql("""
    SELECT tititu, COUNT(*) AS prestamos
    FROM prestamos
    GROUP BY tititu
    ORDER BY prestamos DESC
    LIMIT 20
""").show(truncate=False)

---

## 6. Filtrar por tipo de soporte (solo libros) con DataFrame API

El campo `prcocs` indica el tipo de soporte. Filtramos por `LIB` (libros físicos) para ver cuáles son los más prestados excluyendo DVDs, CDs, etc.

In [ ]:
from pyspark.sql.functions import trim

# Filtrar solo libros físicos (prcocs == 'LIB')
# Nota: los valores del CSV tienen espacios al final, por eso usamos trim()
libros_fisicos = (
    df.filter(trim(col("prcocs")) == "LIB")
      .groupBy("tititu")
      .agg(count("*").alias("prestamos"))
      .orderBy(desc("prestamos"))
)

print("Top 20 libros físicos más prestados:")
libros_fisicos.show(20, truncate=False)

---

## 7. Filtrar por tipo de soporte con Spark SQL

La misma consulta expresada en SQL puro con la cláusula `WHERE`:

In [ ]:
spark.sql("""
    SELECT tititu, COUNT(*) AS prestamos
    FROM prestamos
    WHERE TRIM(prcocs) = 'LIB'
    GROUP BY tititu
    ORDER BY prestamos DESC
    LIMIT 20
""").show(truncate=False)

---

## 8. Ejercicio 1: Tipos de soporte.

Utilizando DataFrames o Spark SQL, muestra cuántos préstamos de cada tipo de soporte se han realizado durante el año.

In [ ]:
# TODO

---

## 9. Ejercicio 2: Préstamos de otros tipos de soportes.

Utilizando DataFrames o Spark SQL, muestra los títulos de DVDs (`DVV`) y Blu-rays (`VBR`) más populares.

In [ ]:
# TODO

---

## 10. Ejercicio 3: Analizar préstamos por sucursal

¿Cuáles son las sucursales de biblioteca con más préstamos? Primero agrupamos por el código de sucursal (`prprsu`) y después mapeamos cada código a su nombre real. Como algunas bibliotecas tienen varios códigos (p.ej. municipal y comunitario), reagrupamos por nombre para consolidar los totales.

In [ ]:
# TODO

In [ ]:
from pyspark.sql.functions import create_map, lit, col, sum as spark_sum

# Diccionario de códigos de sucursal a nombres reales
sucursales = {
    "BCDEPO": "BPM. Depósito general BBPP",
    "BCDI10": "BPM Biblioteca 10",
    "BCDI2": "Manuel Vázquez Montalbán",
    "BCDI3": "Ángel González",
    "BCDI4": "Iván de Vargas",
    "BCDI5": "Eugenio Trías. Casa de Fieras de El Retiro",
    "BCDI6": "BPM Biblioteca 6",
    "BCDI7": "Mario Vargas Llosa",
    "BCDI8": "María Lejárraga",
    "BCDI9": "BPM Biblioteca 9",
    "BCDIA": "Huerta de la Salud",
    "BCDIB": "Ana Mª Matute",
    "BCDIC": "Canillejas",
    "BCDICL": "BPM Club de lectura",
    "BCDID": "La Elipa",
    "BCDID6": "BPM Despachos Unidad Central",
    "BCDIE": "Vicálvaro",
    "BCDIF": "Francisco Ibáñez",
    "BCDIG": "Gloria Fuertes",
    "BCDIH": "Miguel Delibes",
    "BCDII": "Gabriel García Márquez",
    "BCDIJ": "Portazgo",
    "BCDIK": "Aluche",
    "BCDIL": "Pablo Neruda",
    "BCDIM": "La Chata",
    "BCDIN": "Buenavista",
    "BCDIO": "San Blas",
    "BCDIP": "Francisco Ayala",
    "BCDIQ": "Pozo del Tío Raimundo",
    "BCDIR": "José Saramago",
    "BCDIS": "Ciudad Lineal",
    "BCDIT": "María Zambrano",
    "BCDIU": "Pío Baroja",
    "BCDIV": "Vallecas",
    "BCDIW": "José Hierro",
    "BCDIX": "Conde Duque",
    "BCDIY": "Gerardo Diego",
    "BCDIZ": "Dámaso Alonso",
    "BP01": "Unidad Central",
    "BP02": "Acuña",
    "BP03": "Canillejas",
    "BP04": "Luis Rosales",
    "BP05": "José Luis Sampedro",
    "BP06": "Pedro Salinas",
    "BP08": "Rafael Alberti",
    "BP09": "Hortaleza",
    "BP10": "Antonio Mingote",
    "BP11": "Manuel Alvar",
    "BP13": "Moratalaz",
    "BP14": "Paco Rabal",
    "BP16": "Elena Fortún",
    "BP17": "Ruiz Egea",
    "BP18": "José Hierro",
    "BP19": "Miguel Hernández",
    "BP20": "Luis Martín Santos",
    "BP21": "María Moliner",
    "BPM": "Bibliotecas Públicas Municipales de Madrid",
}

# Crear expresión de mapeo código → nombre usando create_map
mapping_expr = create_map([lit(x) for item in sucursales.items() for x in item])

# Mapear código a nombre y reagrupar por nombre para sumar códigos de la misma biblioteca
# (p.ej. "Canillejas" tiene códigos BCDIC y BP03, "José Hierro" tiene BCDIW y BP18)
prestamos_sucursal_nombres = (
    prestamos_sucursal
      .withColumn("sucursal", mapping_expr[col("sucursal")])
      .groupBy("sucursal")
      .agg(spark_sum("prestamos").alias("prestamos"))
      .orderBy(col("prestamos").desc())
)

prestamos_sucursal_nombres.show(40, truncate=False)

**¿Por qué los resultados de la primera celda, con los códigos de las bibliotecas, son diferentes de los de la segunda celda, que usan los nombres de las bibliotecas?**

---

## 11. Ejercicio 4: Analizar préstamos a lo largo del tiempo

Extraemos el mes de la fecha de préstamo (`prfpre`) para ver la distribución temporal de los préstamos a lo largo del año. Después visualizamos el resultado con un gráfico de barras usando matplotlib.

In [ ]:
# TODO

In [ ]:
import matplotlib.pyplot as plt

# Convertir el DataFrame Spark a Pandas para graficar
pdf = prestamos_por_mes.toPandas().dropna().sort_values("mes")

meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
         "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pdf["nombre_mes"] = pdf["mes"].astype(int).map(lambda m: meses[m - 1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(pdf["nombre_mes"], pdf["prestamos"])
ax.set_xlabel("Mes")
ax.set_ylabel("Nº de préstamos")
ax.set_title("Préstamos por mes en bibliotecas de Madrid (2024)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

**¿Se observa algún patrón estacional en los préstamos?**

---

## 12. Cerrar la sesión de Spark

In [ ]:
spark.stop()